# Measured disk hit counts in R and phi

This count-only view uses the extracted mean `(x, y)` of each primary's associated hits on a disk. It counts each generated primary once per disk, at that mean hit position. It does not project missing particles and does not show a hit fraction. The count therefore represents hit-associated primaries, not raw SimTrackerHit multiplicity.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / 'plot_disk_hit_r_phi_counts.py').is_file():
    SCRIPT_DIR = Path('proj_svt_disk_tiling/simrec/acc_study').resolve()
if not (SCRIPT_DIR / 'plot_disk_hit_r_phi_counts.py').is_file():
    raise FileNotFoundError('Run from this notebook directory or the EPIC repository root.')

EXTRACTED_ROOT = SCRIPT_DIR / 'extracted' / 'acceptance_v1'
OUTPUT_TAG = 'hit_r_phi_counts_v1'
OUTPUT_DIR = SCRIPT_DIR / 'results' / OUTPUT_TAG
VERTICES_MM = [0]                 # Add -10 and +10 when those shards exist.
MOMENTUM_BINS_GEV = [
    (0.5, 1.5),
    (1.5, 3.0),
    (3.0, 5.0),
    (5.0, 10.0),
    (10.0, 20.01),                # Include the generated endpoint at 20 GeV.
]
R_MAX_MM = 450.0
R_BIN_WIDTH_MM = 10.0
RUN_SCAN = False                 # Set True to generate the maps.

In [ ]:
command = [
    sys.executable, str(SCRIPT_DIR / 'plot_disk_hit_r_phi_counts.py'),
    '--extracted-root', str(EXTRACTED_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--vertices', *map(str, VERTICES_MM),
    '--r-max-mm', str(R_MAX_MM),
    '--r-bin-width-mm', str(R_BIN_WIDTH_MM),
]
for low, high in MOMENTUM_BINS_GEV:
    command.extend(['--momentum-bin', str(low), str(high)])
print(' '.join(command))
if RUN_SCAN:
    subprocess.run(command, check=True)
else:
    print('Scan not run. Set RUN_SCAN = True when ready.')

## Review the first momentum interval

Each page arranges ED0–ED4 in the left column and HD0–HD4 in the right. All disks use the same R range and bin width for direct visual comparison.

In [ ]:
first_low, first_high = MOMENTUM_BINS_GEV[0]
tag = f'p{first_low:g}_to_{first_high:g}'.replace('.', 'p')
vertex_tag = '000' if VERTICES_MM[0] == 0 else f"{'p' if VERTICES_MM[0] > 0 else 'm'}{abs(VERTICES_MM[0]):03d}"
plot_path = OUTPUT_DIR / f'hit_r_phi_vz_{vertex_tag}mm_{tag}_counts.png'
if plot_path.is_file():
    display(Image(filename=str(plot_path)))
    summary = pd.read_csv(OUTPUT_DIR / 'hit_r_phi_counts_summary.csv')
    selected = (summary.vertex_z_mm == VERTICES_MM[0]) & (summary.momentum_min_GeV == first_low)
    display(summary.loc[selected, ['disk', 'selected_generated_primaries',
                                   'primaries_with_disk_hit', 'primaries_binned_in_r_phi',
                                   'primaries_outside_r_range']])
else:
    print(f'No count map yet. Run the scan to create {OUTPUT_DIR}.')